In [1]:
import pandas as pd

roll_number = "1024170009"
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

# Personalized entry for digit 0 -> billing
personalized_1 = {
    "question": "how can i check my billing history",
    "answer": "You can view your billing history from the Billing section.",
    "keywords": "billing history invoice charges",
    "category": "billing"
}

# Personalized entry for digit 9 -> billing
personalized_2 = {
    "question": "can i get a receipt for my payment",
    "answer": "Yes, a payment receipt is available after the transaction.",
    "keywords": "receipt payment transaction bill",
    "category": "billing"
}

# Create DataFrame
entries = fixed_entries + [personalized_1, personalized_2]

df = pd.DataFrame(entries)

print("========== Q1: FINAL KNOWLEDGE BASE ==========")
print(df)
print()


========== Q1: FINAL KNOWLEDGE BASE ==========
                             question  \
0              what is the annual fee   
1               how to reset password   
2         what are your working hours   
3               how can i pay the fee   
4  how can i check my billing history   
5  can i get a receipt for my payment   

                                              answer  \
0                          The annual fee is Rs 500.   
1                   Go to Settings > Reset Password.   
2                          We are open 9 AM to 5 PM.   
3         You can pay via UPI, card, or net banking.   
4  You can view your billing history from the Bil...   
5  Yes, a payment receipt is available after the ...   

                           keywords category  
0             fee cost price charge  billing  
1              password reset login  account  
2            hours timing open time  general  
3               pay payment upi fee  billing  
4   billing history invoice charges  

In [2]:
# Q2:

def score_query(query, df):
    query_words = set(query.lower().split())

    results = []

    for index, row in df.iterrows():

        question_words = set(row["question"].lower().split())
        keyword_words = set(row["keywords"].lower().split())
        
        matched_question = query_words & question_words
        matched_keywords = query_words & keyword_words

        score = len(matched_question) + len(matched_keywords)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })
            
    results.sort(key=lambda x: x["score"], reverse=True)
    return results

print("========== Q2: SCORING ==========")

query = input("Enter a query: ")

results = score_query(query, df)

if results:
    print("\nMatching entries:")
    for result in results:
        print("Question :", result["question"])
        print("Answer   :", result["answer"])
        print("Category :", result["category"])
        print("Score    :", result["score"])
        print()
else:
    print("No matching FAQ found.")

print()


========== Q2: SCORING ==========


Enter a query:  billing



Matching entries:
Question : how can i check my billing history
Answer   : You can view your billing history from the Billing section.
Category : billing
Score    : 2




In [3]:
# Q3: Find all questions belonging to a category

def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]

personalized_category = personalized_1["category"]

print("========== Q3: SAME CATEGORY ==========")
print("Category:", personalized_category)

category_entries = same_category(personalized_category, df)

print(category_entries[["question", "category"]])
print()


========== Q3: SAME CATEGORY ==========
Category: billing
                             question category
0              what is the annual fee  billing
3               how can i pay the fee  billing
4  how can i check my billing history  billing
5  can i get a receipt for my payment  billing



In [4]:
# Q4: Add a new keyword and save DataFrame to CSV

print("========== Q4: ADD NEW KEYWORD ==========")

entry_index = 0

print("Selected FAQ:")
print(df.loc[entry_index, "question"])

new_keyword = input("Enter a new keyword to add: ")
df.loc[entry_index, "keywords"] = (
    df.loc[entry_index, "keywords"] + " " + new_keyword
)

print("\nUpdated entry:")
print(df.loc[entry_index])

filename = roll_number + "_faq_data.csv"
df.to_csv(filename, index=False)

print("\nUpdated DataFrame saved as:", filename)
print()

========== Q4: ADD NEW KEYWORD ==========
Selected FAQ:
what is the annual fee


Enter a new keyword to add:  annual



Updated entry:
question          what is the annual fee
answer         The annual fee is Rs 500.
keywords    fee cost price charge annual
category                         billing
Name: 0, dtype: str

Updated DataFrame saved as: 1024170009_faq_data.csv



In [7]:
# Q5: Count FAQ entries per category using groupby

print("========== Q5: FAQ COUNT PER CATEGORY ==========")

category_counts = df.groupby("category").count()

print(category_counts)
print()


========== Q5: FAQ COUNT PER CATEGORY ==========
          question  answer  keywords
category                            
account          1       1         1
billing          4       4         4
general          1       1         1



In [ ]:

# ============================================================
# Q6: Modified scoring function with tie handling
# ============================================================

def score_query_with_ties(query, df):

    query_words = set(query.lower().split())

    results = []

    for index, row in df.iterrows():

        question_words = set(row["question"].lower().split())
        keyword_words = set(row["keywords"].lower().split())

        matched_question = query_words & question_words
        matched_keywords = query_words & keyword_words

        score = len(matched_question) + len(matched_keywords)

        if score > 0:
            results.append({
                "index": index,
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "score": score
            })

    if not results:
        print("No matching entries found.")
        return []

    # Highest score
    max_score = max(result["score"] for result in results)

    # Get ALL entries having highest score
    best_matches = [
        result for result in results
        if result["score"] == max_score
    ]

    print("\nHighest confidence score:", max_score)

    if len(best_matches) > 1:
        print("TIE DETECTED! Multiple entries have the same highest score.\n")

    for result in best_matches:
        print("Question :", result["question"])
        print("Answer   :", result["answer"])
        print("Category :", result["category"])
        print("Score    :", result["score"])
        print()

    return best_matches

print("========== Q6: TIE DEMONSTRATION ==========")

tie_query = "fee"

print("Query:", tie_query)

score_query_with_ties(tie_query, df)


print("========== Q6: NON-TIE DEMONSTRATION ==========")

non_tie_query = "password"

print("Query:", non_tie_query)

score_query_with_ties(non_tie_query, df)